# 🔍 Financial Crime Intelligence — Day 1

**Fraud Detection Application** — Rule-based risk scoring with interactive Gradio UI and AI Investigation Reports.

This notebook implements a complete fraud detection pipeline:
1. Upload a CSV of transactions
2. Apply rule-based fraud scoring
3. View risk-scored results, summary metrics, and a distribution chart
4. **[NEW]** Generate AI investigation reports for HIGH risk transactions using a local LLM.

---

## Section 1: Package Installation

Install all required dependencies. This cell is safe to re-run — pip will skip already-installed packages.

In [ ]:
# ============================================================
# Section 1: Package Installation
# ============================================================

!pip install --quiet pandas numpy matplotlib gradio transformers torch accelerate

print("✅ All packages installed successfully.")

## Section 2: Imports

Import all libraries needed for data processing, visualization, the interactive UI, and the local LLM.

In [ ]:
# ============================================================
# Section 2: Imports
# ============================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import gradio as gr
import tempfile
import os

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("\n✅ All imports loaded.")

## Section 3: Fraud Detection Logic

Define the rule-based scoring engine. Each transaction is scored on:

| Rule | Condition | Points |
|------|-----------|--------|
| 1 | `amount > 500,000` | +50 |
| 2 | `amount > 1,000,000` | +20 (additional) |
| 3 | `country` in high-risk list | +30 |

**Risk Levels:** LOW (0–29) · MEDIUM (30–69) · HIGH (70–100)

In [ ]:
# ============================================================
# Section 3: Fraud Detection Logic
# ============================================================

HIGH_RISK_COUNTRIES = ["Russia", "Nigeria", "North Korea"]

def compute_risk_score(row: pd.Series) -> int:
    score = 0
    if row["amount"] > 500_000: score += 50
    if row["amount"] > 1_000_000: score += 20
    if row["country"] in HIGH_RISK_COUNTRIES: score += 30
    return min(score, 100)

def classify_risk_level(score: int) -> str:
    if score >= 70: return "HIGH"
    elif score >= 30: return "MEDIUM"
    else: return "LOW"

print("✅ Fraud detection logic defined.")

## Section 4: Model Loading

Load the local instruction model `Qwen/Qwen3-14B`. 
We load it using `bfloat16` to optimize VRAM usage on AMD ROCm GPUs.

In [ ]:
# ============================================================
# Section 4: Model Loading
# ============================================================

MODEL_ID = "Qwen/Qwen3-14B"

print(f"⏳ Loading tokenizer and model: {MODEL_ID}...")
print("This may take a few minutes and requires substantial VRAM.")

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Load the model directly to the GPU (device_map="auto") using bfloat16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("✅ Model loaded successfully.")

## Section 5: Prompt Templates

Define the strict system prompt and the user prompt templates for generating the investigation report.

In [ ]:
# ============================================================
# Section 5: Prompt Templates
# ============================================================

SYSTEM_PROMPT = """You are an expert Financial Crime Investigator.
Analyze the given transaction details and provide a concise, professional investigation report.

Strictly format your response using exactly these headings:
- Risk Summary:
- Reason For Alert:
- Potential Risks:
- Recommended Investigation Steps:
- Final Recommendation:
"""

def build_user_prompt(amount, country, risk_score, risk_level):
    return f"""Transaction Details:
- Amount: {amount}
- Country: {country}
- Risk Score: {risk_score}
- Risk Level: {risk_level}

Please generate the investigation report."""

print("✅ Prompt templates defined.")

## Section 6: Report Generation

Use the loaded model to generate reports only for HIGH risk transactions.

In [ ]:
# ============================================================
# Section 6: Report Generation
# ============================================================

def generate_investigation_report(amount, country, risk_score, risk_level):
    """Generates a single investigation report using the LLM."""
    user_prompt = build_user_prompt(amount, country, risk_score, risk_level)
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]
    
    # Apply chat template if available, otherwise just concatenate
    if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template is not None:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = f"{SYSTEM_PROMPT}\n\n{user_prompt}\n\nReport:\n"
        
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # Generate output
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    # Decode only the newly generated tokens
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()


def batch_generate_reports(df: pd.DataFrame):
    """Filters for HIGH risk transactions and batches report generation."""
    if df is None or df.empty:
        return "No data available to analyze."
        
    high_risk_df = df[df["risk_level"] == "HIGH"]
    
    if high_risk_df.empty:
        return "✅ **No HIGH risk transactions found.** AI Investigation Reports are only generated for HIGH risk levels."
    
    reports_md = ["## 🤖 AI Investigation Reports\n"]
    
    for _, row in high_risk_df.iterrows():
        tx_id = row.get("transaction_id", "Unknown")
        amount = row.get("amount", 0)
        country = row.get("country", "Unknown")
        risk_score = row.get("risk_score", 0)
        risk_level = row.get("risk_level", "HIGH")
        
        reports_md.append(f"### Transaction: {tx_id}")
        try:
            report = generate_investigation_report(amount, country, risk_score, risk_level)
            reports_md.append(report)
        except Exception as e:
            reports_md.append(f"*Error generating report: {str(e)}*")
            
        reports_md.append("---")
        
    return "\n\n".join(reports_md)

print("✅ Report generation logic defined.")

## Section 7: Analysis Function

The main analysis pipeline for parsing CSV and calculating scores.

In [ ]:
# ============================================================
# Section 7: Analysis Function
# ============================================================

REQUIRED_COLUMNS = ["transaction_id", "account", "amount", "country", "timestamp"]

def generate_risk_chart(df: pd.DataFrame) -> str:
    risk_order = ["LOW", "MEDIUM", "HIGH"]
    counts = df["risk_level"].value_counts().reindex(risk_order, fill_value=0)
    colors = ["#2ecc71", "#f39c12", "#e74c3c"]
    fig, ax = plt.subplots(figsize=(8, 5), facecolor="#1a1a2e")
    ax.set_facecolor("#1a1a2e")
    bars = ax.bar(counts.index, counts.values, color=colors, edgecolor="white", linewidth=0.8, width=0.6)
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2.0, height + 0.3, f"{int(height)}", ha="center", va="bottom", fontweight="bold", fontsize=14, color="white")
    ax.set_title("Risk Level Distribution", fontsize=18, fontweight="bold", color="white", pad=15)
    ax.set_xlabel("Risk Level", fontsize=13, color="white", labelpad=10)
    ax.set_ylabel("Number of Transactions", fontsize=13, color="white", labelpad=10)
    ax.tick_params(axis="both", colors="white", labelsize=12)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#444")
    ax.spines["bottom"].set_color("#444")
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    plt.tight_layout()
    chart_path = os.path.join(tempfile.gettempdir(), "risk_distribution.png")
    fig.savefig(chart_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    return chart_path

def analyze_transactions(file):
    if file is None:
        return ("⚠️ No file uploaded.", pd.DataFrame(), None)
    try:
        df = pd.read_csv(file)
    except Exception as e:
        return (f"❌ Error reading CSV: {e}", pd.DataFrame(), None)
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        return (f"❌ Missing columns: {', '.join(missing)}\nExpected: {', '.join(REQUIRED_COLUMNS)}", pd.DataFrame(), None)
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce").fillna(0)
    df["country"] = df["country"].astype(str).str.strip()
    df["risk_score"] = df.apply(compute_risk_score, axis=1)
    df["risk_level"] = df["risk_score"].apply(classify_risk_level)
    total = len(df)
    high_count = int((df["risk_level"] == "HIGH").sum())
    medium_count = int((df["risk_level"] == "MEDIUM").sum())
    low_count = int((df["risk_level"] == "LOW").sum())
    summary = (f"📊  SUMMARY METRICS\n{'═' * 40}\n  Total Transactions : {total}\n  🔴 High Risk       : {high_count}\n  🟡 Medium Risk     : {medium_count}\n  🟢 Low Risk        : {low_count}\n{'═' * 40}")
    results_df = df[["transaction_id", "account", "amount", "country", "risk_score", "risk_level"]].copy()
    chart_path = generate_risk_chart(df)
    return summary, results_df, chart_path

print("✅ Analysis function defined.")

## Section 8: UI Integration

Build the interactive UI:
- **Upload** a CSV file
- **Click Analyze** to run the fraud scoring pipeline
- **Click Generate AI Investigation Reports** to run the LLM inference on HIGH risk transactions
- **View** the summary, results table, charts, and AI reports

In [ ]:
# ============================================================
# Section 8: UI Integration
# ============================================================

theme = gr.themes.Base(primary_hue="blue", neutral_hue="slate", font=gr.themes.GoogleFont("Inter"))
demo = gr.Blocks(theme=theme, title="Financial Crime Intelligence")

with demo:
    gr.Markdown("""# 🛡️ Financial Crime Intelligence\n### Rule-Based Fraud Detection & AI Investigator Engine\nUpload a CSV of transactions to analyze, then generate AI reports for High-Risk flags.""")
    with gr.Row():
        with gr.Column(scale=1):
            csv_upload = gr.File(label="📁 Upload Transaction CSV", file_types=[".csv"], type="filepath")
            analyze_btn = gr.Button("🔍 Analyze Transactions", variant="primary", size="lg")
            gr.Markdown("""**Scoring Rules:**\n- Amount > 500K → +50 pts\n- Amount > 1M → +20 pts\n- High-risk country → +30 pts""")
        with gr.Column(scale=2):
            summary_output = gr.Textbox(label="📊 Summary Statistics", lines=8, interactive=False)
    
    gr.Markdown("### 📋 Scored Transactions")
    results_table = gr.Dataframe(label="Results", headers=["transaction_id", "account", "amount", "country", "risk_score", "risk_level"], interactive=False)
    
    with gr.Row():
        generate_ai_btn = gr.Button("🤖 Generate AI Investigation Reports", variant="secondary", size="lg")
        
    ai_reports_output = gr.Markdown(label="AI Investigation Reports")
    
    gr.Markdown("### 📈 Risk Distribution Chart")
    chart_output = gr.Image(label="Risk Level Distribution", type="filepath")

    # Wire up the analyze button
    analyze_btn.click(fn=analyze_transactions, inputs=[csv_upload], outputs=[summary_output, results_table, chart_output])
    
    # Wire up the AI Generation button
    generate_ai_btn.click(fn=batch_generate_reports, inputs=[results_table], outputs=[ai_reports_output])

print("✅ Gradio interface built.")

## Section 9: Launch Application

Start the Gradio server.

In [ ]:
# ============================================================
# Section 9: Launch Application
# ============================================================

demo.launch(inline=True, share=False)